# Decomposição do erro quadrático em regressão polinomial

Considere o modelo gerador

$$
Y = f(x) + \varepsilon,
\qquad \varepsilon \sim \mathcal{N}(0,1)
$$

com

$$
f(x) = 5 x^2 (1 - x).
$$

O objetivo deste exercício é estudar, na prática, a decomposição do erro quadrático médio de predição em três partes:

$$
\mathbb{E}\big[(Y - \hat f(x))^2\big]
=
\underbrace{\text{erro irredutível}}_{\sigma^2}
+
\underbrace{\text{viés}^2}_{(\mathbb{E}[\hat f(x)] - f(x))^2}
+
\underbrace{\text{variância}}_{\mathbb{V}(\hat f(x))}.
$$

## Exercício 1

1. Gere $M = 500$ amostras de treinamento independentes.
2. Cada amostra deve ter $n = 100$ observações.
3. Em cada amostra:
   - sorteie $x \sim \text{Uniforme}(-2, 2)$;
   - gere $y = f(x) + \varepsilon$, com $\varepsilon \sim \mathcal{N}(0,1)$;
   - ajuste modelos de regressão polinomial com graus $1$, $2$, $5$ e $10$.
4. Considere uma grade fixa com 200 pontos no intervalo $[-2,2]$.
5. Para cada grau do polinômio e para cada ponto da grade, estime:
   - a média das predições;
   - o viés ao quadrado;
   - a variância das predições.
6. Estime empiricamente também o erro irredutível.
7. Calcule, para cada grau:
   - a média do viés ao quadrado na grade;
   - a média da variância na grade;
   - a estimativa do erro irredutível;
   - a soma dos três componentes;
8. Troque os valores de $M, n$ e os graus dos polinômios e compare os comportamentos.



### Dicas

Sejam $\hat f^{(1)}(x), \hat f^{(2)}(x), \dots, \hat f^{(M)}(x)$ as predições obtidas a partir das $M$ amostras de treinamento.

- A média das predições em um ponto $x$ é dada por: $\overline{\hat f}(x) = \frac{1}{M}\sum_{m=1}^M \hat f^{(m)}(x).$

- O viés ao quadrado em um ponto $x$ é dado por: $\text{Viés}^2(x) = \left(\overline{\hat f}(x) - f(x)\right)^2.$

- A variância em um ponto $x$ é dada por: $\text{Var}(x) = \frac{1}{M}\sum_{m=1}^M \left(\hat f^{(m)}(x) - \overline{\hat f}(x)\right)^2.$

- Uma estimativa prática do erro irredutível pode ser obtida por: $\widehat{\sigma^2}=\frac{1}{N}\sum_{i=1}^N \left(y_i - f(x_i)\right)^2,$

Onde $N$ é o número total de observações geradas ao longo das amostras.

### Funções úteis em Python

Para ajustar um modelo polinomial com `scikit-learn`, você pode usar `PolynomialFeatures` e `LinearRegression`.

Exemplo:

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=grau, include_bias=True)
X_train_poly = poly.fit_transform(x_train)
X_grid_poly = poly.transform(x_grid)

modelo = LinearRegression(fit_intercept=False)
modelo.fit(X_train_poly, y_train)

# Predição na grade
y_pred_grid = modelo.predict(X_grid_poly)
preds.append(y_pred_grid)

## Exercício 2

Neste exercício, em vez de repetir muitas amostras para estimar viés, variância e erro irredutível, você vai trabalhar com **uma única nova amostra** de tamanho $n$ e usá-la para fazer **seleção de modelo**.

1. Gere uma nova amostra de tamanho $n$ a partir do modelo anterior.
2. Separe essa amostra em três partes:
   - **treino**;
   - **calibração**;
   - **teste**.
3. Ajuste modelos de regressão polinomial com os mesmos graus do Exercício 1:
   - grau $1$;
   - grau $2$;
   - grau $5$;
   - grau $10$.
4. Use apenas o conjunto de **treino** para ajustar os modelos.
5. Avalie os modelos no conjunto de **calibração** e escolha o grau que apresentar o menor erro quadrático médio nesse conjunto.
6. Depois de escolher o melhor grau com base na calibração, avalie esse modelo no conjunto de **teste**.

7. Compare:
   - o grau escolhido pela calibração;
   - o erro obtido no teste;
   - os resultados observados no Exercício 1.

### Funções úteis em Python

Para gerar a amostra e separar em treino, calibração e teste:

```python
from sklearn.model_selection import train_test_split

# primeiro separa teste
x_temp, x_teste, y_temp, y_teste = train_test_split(
    x, y, test_size=0.2, random_state=123
)

# depois separa treino e calibração
x_treino, x_cal, y_treino, y_cal = train_test_split(
    x_temp, y_temp, test_size=0.25, random_state=123
)
```


## Exercício 3

Neste exercício, você vai substituir a divisão em treino, calibração e teste por um procedimento de **validação cruzada** para escolher o grau do polinômio.

1. Gere uma nova amostra de tamanho $n$ a partir do modelo anterior.

2. Separe essa amostra em duas partes:
   - **treino**;
   - **teste**.

3. Considere os mesmos graus polinomiais dos exercícios anteriores:
   - grau $1$;
   - grau $2$;
   - grau $5$;
   - grau $10$.

4. Use apenas o conjunto de **treino** para realizar a escolha do modelo por **validação cruzada**.

5. Para cada grau:
   - aplique validação cruzada no conjunto de treino;
   - calcule o erro quadrático médio médio entre os folds.

6. Escolha o grau que apresentar o menor erro médio de validação cruzada.

7. Reajuste o modelo escolhido usando **todo o conjunto de treino**.

8. Avalie o modelo final no conjunto de **teste**.

9. Compare:
   - o grau escolhido por validação cruzada;
   - o erro estimado pela validação cruzada;
   - o erro observado no conjunto de teste;
   - os resultados obtidos nos Exercícios 1 e 2.


### Funções úteis em Python

Para criar explicitamente os índices de treino e validação em cada fold:

```python
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=123)

for fold, (idx_treino, idx_validacao) in enumerate(cv.split(x_treino), start=1):
    x_fold_treino = x_treino[idx_treino]
    y_fold_treino = y_treino[idx_treino]

    x_fold_validacao = x_treino[idx_validacao]
    y_fold_validacao = y_treino[idx_validacao]